<a href="https://colab.research.google.com/github/GreatAmus/boardgame_recommender/blob/main/Board_Game_Recommendation_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Final Project**
## *DATA 5420/6420*
## Name: Jeremy Rowley

The purpose of the final project is to produce an MVP that is a culmination of the skills you have learned in each of the previous units. This MVP should be a cohesive product in that it combines methods in some logical pipeline, it should NOT simply be a collection of methods implemented independently/separately with no clear end goal/state. You will be tasked with applying at least four methods from across the four units, which I've outlined below:

### Unit 1

* Chatbots
* Basic Text Statistics
* NLP Pipeline (Preprocessing & Normalization)
* Compiling Corpora via APIs

### Unit 2

* Bag of Words Models (TF-IDF and Count Vectorization)
* Document Classification
* Sentiment Analysis

### Unit 3

* Document Summarization
* Topic Modeling
* Text Similarity
> * Information Retrieval (Search)
> * Recommendation Systems
* Document Clustering
> * KMeans
> * Affinity Prop
> * Wards Agglomerative Hierarchical

### Unit 4

* Word Embeddings
* Pretrained Transformers
* Question-Answering Systems
* Speech-to-Text (hopefully)

You will of course need to perform some form of cleaning/text normalization and feature engineering (bag of words and/or word embeddings), but the way you go about that will be problem dependent -- on top of those two steps, you will need to incorporate at least two other model types as well that form some coherent end-stage MVP.

For example:

1) corpus of a news articles pulled from the Bing News API that is cleaned/normalized

2) uses word embeddings to feature engineer the text

3) performs sentiment analysis to score sentiment of all articles

4) articles are sortable by sentiment, and ranked based on their relevance to keywords/search queries (information retrieval)

The MVP is a NewsFeed showing a table of articles displayed in an interactive dashboard

As you are performing your analyses consider:

* What cleaning and normalization steps are necessary for my text, and which are not?
* What sort of feature engineering do I need to utilize, both in terms of using BoW or word embeddings, and in terms of document or word vectorization? Do I need to use different methods for different analysis types?
* What is the purpose of performing your selected methods and how do they meaningful build on one another?
* What are the practical applications of the models you developed?

### **What four (+) methods have you chosen and how do they fit together?**



1.   List item
2.   List item
3.
4.

**Description of how these methods will be meaningfully combined**:



### The rest of this notebook is free form. Make sure it is clear what steps are being taken at each stage of your project, comments and section headers are recommended!

Think about if you were to share this work with a future employer or another future co-worker, they should be able to understand your logic and the steps you've taken.

BoardGameGeek rating data can be obtained from the following link:
https://boardgamegeek.com/data_dumps/bg_ranks

## Project roadmap (easy to follow)
1. Setup & imports
2. Method 0 — Data pull (BGG XML API) *(copied from Unit 3)*
3. Corpus building *(Unit 3: one document per game)*
4. Cleaning / normalization *(Unit 3 boilerplate removal + lemmatization)*
5. Feature engineering *(Unit 3: TF‑IDF → SVD → Normalize)*
6. Method 1 — Clustering *(Unit 3: KMeans)*
7. Method 2 — Similarity recommendations *(Unit 3: cosine similarity)*
8. Method 3 — Sentiment *(VADER)*
9. Method 4 — LLM explanations *(Gemini/OpenAI optional)*
10. Persist artifacts for the Streamlit MVP

> **Notebook rule:** each top-level `def` / `class` is placed in its own cell.


In [1]:
# Installs
!pip install langdetect
!pip -q install streamlit joblib pyarrow scikit-learn
#!pip install google-genai # Install the new Gemini package

# Core
import os
import re
import math
import time
import html

# API
from langdetect import detect, DetectorFactory
import requests
import xml.etree.ElementTree as ET
import json

# Data
import numpy as np
import pandas as pd

# NLP
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.tokenize import ToktokTokenizer
from nltk.corpus import stopwords

#os.environ["THINC_NO_TORCH"] = "1"
#os.environ["CUDA_VISIBLE_DEVICES"] = ""

nltk.download('stopwords')
nltk.download('punkt_tab')
import spacy
nlp = spacy.load("en_core_web_sm")
lemmatizer = nlp.get_pipe("lemmatizer")
tokenizer = ToktokTokenizer()

# Sentiment Analysis
from nltk.sentiment import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')

# ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import make_pipeline

# Streamlit
import joblib

# Gemini
#import google.genai as genai # Use the new package


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [2]:
class SavedData():
  def __init__(self):
      self._file = 'bgg_forum_reviews.csv' # Retrieve BGG review data. Expected format is game_name and review_text columns
      self._dataframe = None # data frame with data
      self._tv = None # TfidfVectorizer
      self._svd = None # Truncated singular value decomposition
      self._norm = None  # Normalizer from sklearn.preprocessing
      self._titles = None # Used for cosine similarity
      self._similarity = None # Cosing similarity dataframe
      self.cluster_labels = {
        0: "Tile-laying and spatial puzzles",
        1: "Air & naval wargames",
        2: "Conflict strategy and dudes-on-a-map",
        3: "Kids & family games",
        4: "Deduction & puzzles",
        5: "Word games",
        6: "Land wargames",
        7: "Dungeon crawler",
        8: "Light filler and abstract games",
        9: "Heavy euro and engine building",
        10: "Midweight tiel and action games (mid Euros)",
        11: "Card games",
    }

  @property
  def file(self):
    return self._file

  @property
  def dataframe(self):
    return self._dataframe
  @dataframe.setter
  def dataframe(self, value):
    self._dataframe = value

  @property
  def tv(self):
    return self._tv
  @tv.setter
  def tv(self, value):
    self._tv = value

  @property
  def svd(self):
    return self._svd
  @svd.setter
  def svd(self, value):
    self._svd = value

  @property
  def norm(self):
    return self._norm
  @norm.setter
  def norm(self, value):
    self._norm = value

  @property
  def titles(self):
    return self._titles
  @titles.setter
  def titles(self, value):
    self._titles = value

  @property
  def similarity(self):
    return self._similarity
  @similarity.setter
  def similarity(self, value):
    self._similarity = value

In [3]:
# Globals
from google.colab import userdata
BGG_FORUMLIST_URL = "https://boardgamegeek.com/xmlapi2/forumlist"
BGG_FORUM_URL     = "https://boardgamegeek.com/xmlapi2/forum"
BGG_THREAD_URL    = "https://boardgamegeek.com/xmlapi2/thread"
API_KEY = userdata.get('BGG')

project_data = SavedData()

## Method 0 — Data pull via BGG XML API (from Unit 3)
Use this to repull the data if needed. Right now, I'm just using a saved CSV from Unit 3.

In [4]:
# For this project, I want to make sure I grab both the top games and a random sample of games.
# Grabbing both will give me a set of good games to compare to other games.
def get_sample(df, top_count = 100, random_count = 800, min_users = 50):

  # I need the data to be numeric or else it brings in the to games of "1", "10", "100", etc
  df["rank"] = pd.to_numeric(df["rank"])
  df["id"] = pd.to_numeric(df["id"])
  df["usersrated"] = pd.to_numeric(df["usersrated"])

  # Exclude expansions and where the rank is not zero (which are unranked games in BGG)
  filtered_games = df[
        (df["is_expansion"] == 0) &
        (df["rank"].notna()) &
        (df["rank"] > 0) &
        (df["id"].notna())
    ].copy()

  # Get the top rated games
  top = filtered_games.sort_values(["rank"]).head(top_count)

  # Get the required sample of games, filtering out the top games.
  # I added a filter for a minimum number of users. Games with fewer users some times have reviews written by the friend of the designer.
  pool = filtered_games[(filtered_games["usersrated"] > min_users) & (~filtered_games["id"].isin(top["id"]))]
  sample = pool.sample(n=min(random_count, len(pool)))

  # Combine the top and sampled games
  subset = pd.concat([top, sample], ignore_index=True)

  return subset

In [5]:
DetectorFactory.seed = 0
# Turns out a lot of reviews are NOT in English. My first attempt failed because I had so many German, Dutch, and French reviews.
# Skip any non-English reviews. Thanks ChatGPT for the help here
def is_english(text, min_len: int = 80):
    t = text.strip()
    if len(t) < min_len:
        return False  # too short so skip it (pretend it isn't English)
    try:
        return detect(t) == "en"
    except Exception:
        return False

In [6]:
# This is my API pulling code. I use a class as it is easier to manager errors that way.
class BGGForumError(Exception):
    pass


In [7]:
# Make a request to BGG with a backoff script to retry in case of throttling
def _api_get(session, url, headers, params, attempts=6,
             base_backoff=1.0, cap_backoff=30.0, throttle_s=0.8):

    # Set up the initial backoff values
    backoff = base_backoff
    last_resp = None

    # Try to pull the BGG info up to the number of attempts allowed. If all of them fail, raise an error
    for _ in range(attempts):

        # Try to get a response
        try:
            resp = session.get(url, params=params, headers=headers, timeout=30)
            last_resp = resp

        # No response so sleep and adjust the backoff time
        except requests.RequestException:
            time.sleep(backoff)
            backoff = min(backoff * 2, cap_backoff)
            continue

        # Good response - return the response immediately
        if resp.status_code == 200:
            return resp

        # Bad response - sleep and then retry
        if resp.status_code in (202, 429, 500, 502, 503, 504):
            retry_after = resp.headers.get("Retry-After")
            if retry_after and retry_after.isdigit():
                time.sleep(float(retry_after))
            else:
                time.sleep(backoff)
                backoff = min(backoff * 2, cap_backoff)
            continue

        raise BGGForumError(f"BGG error: HTTP {resp.status_code} – {resp.text[:200]}")

    status = getattr(last_resp, "status_code", "no_response")
    raise BGGForumError(f"BGG request failed after retries (last status: {status}).")


In [8]:
# Find the forum ID of the game's forum
def get_forum_id(game_id, session, throttle_s = 1.0):
    headers = {"Authorization": f"Bearer {API_KEY}"}
    params = {"type": "thing", "id": str(game_id)}

    resp = _api_get(session, BGG_FORUMLIST_URL, headers=headers, params=params)

    # convert to searchable tree
    root = ET.fromstring(resp.text)

    # Find forum whose title/name contains "review"
    for forum in root.findall(".//forum"):
        title = (forum.attrib.get("title") or forum.attrib.get("name") or "").strip()
        if "review" in title.lower():
            return int(forum.attrib["id"]), title

    return None, None


In [9]:
# Only get the first post on a review. The rest are comments on the review, not the review itself
def get_first_post(thread_id, session):
    headers = {"Authorization": f"Bearer {API_KEY}"}
    params = {"id": str(thread_id), "count": "50"}

    # Call the API and get a review back
    resp = _api_get(session, BGG_THREAD_URL, headers=headers, params=params)
    root = ET.fromstring(resp.text)

    thread_node = root.find(".//thread")
    thread_subject = thread_node.attrib.get("subject") if thread_node is not None else None

    # All forum posts are considered articles - this would be the articles in the "review" thread
    articles = root.findall(".//article")
    if not articles:
        return None

    # Parse each article into a dict (body lives in <body>)
    parsed = []
    for a in articles:
        body_node = a.find("body")
        body = "".join(body_node.itertext()).strip() if body_node is not None else ""

        if not body:
            continue

        # I don't need all of this info, but figured I can stript it out after
        postdate = a.attrib.get("postdate")
        parsed.append({
            "thread_id": thread_id,
            "thread_subject": thread_subject,
            "article_id": int(a.attrib.get("id", "0") or 0),
            "username": a.attrib.get("username"),
            "postdate": postdate,
            "subject": a.findtext("subject", default="") or a.attrib.get("subject"),
            "review_text": body,
        })

    if not parsed:
        return None

    # Choose the earliest postdate or lowest article ID. I sorted by age to keep the process consistent
    parsed.sort(key=lambda x: (x["postdate"] or "9999", x["article_id"]))
    return parsed[0]


In [10]:
# Grab the reviews based on the forum ID found using get_forum_id
def get_forum_threads(forum_id, session, page = 1):
    headers = {"Authorization": f"Bearer {API_KEY}"}
    params = {"id": str(forum_id), "page": str(page)}

    resp = _api_get(session, BGG_FORUM_URL, headers=headers, params=params)
    root = ET.fromstring(resp.text)

    # collect all of the threads
    threads = []
    for t in root.findall(".//thread"):
        threads.append({
            "thread_id": int(t.attrib["id"]),
            "subject": t.attrib.get("subject"),
            "author": t.attrib.get("author"),
        })
    return threads

In [11]:
# The main API driver. The previous functions are helpers.
# This one grabs the forum ID, uses the forum ID to get the threads, then uses the threads to get the article
def get_game_reviews(
    game_id,
    session,
    game_name = None,
    forum_page = 1,
    max_threads = 5,
    min_char = 200,
    throttle_s = 1.0,
    verbose: bool = True,
):

    # Get the forum ID
    forum_id, forum_title = get_forum_id(game_id, session, throttle_s=throttle_s)
    if not forum_id:
        if verbose:
            print(f"[{game_id}] No Reviews forum found.")
        return pd.DataFrame(columns=[
            "game_id","game_name","forum_id","forum_title",
            "thread_id","thread_subject","username","postdate","review_text"
        ])

    # No forum ID - there's no reviews for this game.
    if verbose:
        print(f"[{game_id}] Reviews forum: {forum_title} (forum_id={forum_id})")

    # Try to avoid rate limiting
    time.sleep(throttle_s)

    # Get the threads related to the forum ID
    threads = get_forum_threads(forum_id, session, page=forum_page)[:max_threads]
    if len(threads) < 1:
        if verbose:
            print(f"[{game_id}] No review threads on first page; skipping.")
        return pd.DataFrame(columns=["game_id","game_name","forum_title","thread_subject","postdate","review_text"])

    # Create a list of the reviews
    rows = []
    for t in threads:
        time.sleep(throttle_s)
        op = get_first_post(t["thread_id"], session)
        if op is None:
            if verbose:
                print(f"[{game_id}] Thread {t['thread_id']} -> OP not found/empty")
            continue

        text = (op["review_text"] or "").strip()

        # If it's not English, don't included it
        if not is_english(text, min_len=min_char):
            continue

        # Return the review info
        rows.append({
            "game_id": game_id,
            "game_name": game_name,
            "forum_title": forum_title,
            "thread_subject": op["thread_subject"] or t.get("subject"),
            "postdate": op["postdate"],
            "review_text": op["review_text"],
        })

        if verbose:
            print(f"[{game_id}] Thread {t['thread_id']} -> kept OP by {op['username']}")

    return pd.DataFrame(rows)


In [12]:
def resume_data_pull(output):

  # Does the output file exist or is this a band new query?
  # The last line is usually bad because some of the reviews will be missed.
  if os.path.exists(output) and os.path.getsize(output) > 0:
      try:
          existing = pd.read_csv(output)
      except pd.errors.EmptyDataError:
          existing = pd.DataFrame()
      except Exception:
          existing = pd.read_csv(output, on_bad_lines="skip")

      # Reviews found - read these and add them to our list. first_write alerts the system that we have reviews already
      if not existing.empty and "game_id" in existing.columns:
          done_ids = set(existing["game_id"].unique())
          current_reviews = len(existing)
          first_write = False

      # Can't get the data so let's start over
      else:
          current_reviews = 0
          done_ids = set()
          first_write = True

  # No output file means start fresh!
  else:
      done_ids = set()
      first_write = True
      current_reviews = 0 # Initialize current_reviews here as well
  return done_ids, first_write, current_reviews

In [13]:
# Sometimes I'd hit rate limiting so bad that it would time out the code.
# I use this output file to save my place so I can restart
def get_api_reviews(project_data: SavedData, df_games: pd.DataFrame):

  output = project_data.file

  # I want to have 1000 reviews in my corpus. 1000 seemed like a good number.
  target = 2500

  # get sample data
  df_sample = get_sample(df_games)

  # If it crashes, I want to reload the data and resume parsing
  done_ids, first_write, current_reviews = resume_data_pull(output)


  print(f"Resuming with {current_reviews} reviews already saved and {len(done_ids)} games completed.")

  # Keep the session stable to avoid excessive API calls and increase performance
  session = requests.Session()

  # Thanks ChatGPT! I would definitely not have guessed at the headings here
  session.headers.update({"Accept-Encoding": "gzip, deflate", "User-Agent": "BGG-Forum-Reviews/1.0"})

  # Game count
  total = len(df_sample)
  game_ids = df_sample["id"].astype(int).tolist()
  game_names = df_sample["name"].astype(str).tolist()

  # Initialize df_game to an empty DataFrame to prevent UnboundLocalError
  df_game = pd.DataFrame()

  # Loop over the games and
  for row in range(0, total):
      if current_reviews >= target:
          print(f"Reached TARGET_REVIEWS >= {target}. Stopping.")
          break

      game_id = int(game_ids[row])
      game_name = game_names[row]

      # Skip games in our CSV
      if game_id in done_ids:
          print(f"({row}/{total}) Skipped {game_name} as it was found in the CSV")
          continue

      # Call the API and get the game information. I turned of verbose as that was mostly for debugging
      df_game = get_game_reviews(
          game_id=game_id,
          session=session,
          game_name=game_name,
          max_threads=6,
          min_char = 200,
          forum_page=1,
          throttle_s=1.0,
          verbose=False
      )

      # Only write if we got any reviews so we don't have games without reviews
      if not df_game.empty:
          df_game.to_csv(output, mode="a", header=first_write, index=False)
          first_write = False
          current_reviews += len(df_game)

      done_ids.add(game_id)

      print(f"({row+1}/{total}) Saved {len(df_game)} reviews for {game_name} | total reviews={current_reviews}")
  return df_game

In [14]:
# Load reviews from the CSV file
def load_reviews(file) -> pd.DataFrame:
    df = pd.read_csv(file)
    df['game_name'] = df['game_name'].astype(str).str.strip()
    df['review_text'] = df['review_text'].astype(str).fillna('').str.strip()
    df = df[df['game_name'].str.len() > 0].copy()
    return df


In [15]:
def pull_data(project_data, skip_data = True):
  if skip_data:
    project_data.dataframe = load_reviews(project_data.file)
    print(project_data.dataframe.info())
    project_data.dataframe.shape
  else:
    df_games = pull_data()
    project_data.dataframe = get_api_reviews(project_data, df_games)
    df.head()
  return df_games

In [16]:
# pull_data(project_data, True)

## Method 0.5: Corpus building and cleaning
Combine all of the reviews into one document

In [17]:
'''
Combine all of the reviews into a single, master review
I combined them for a couple of reasons:
1) Some games have 1 review and others hav a lot. Having only one review per game (even if it really is multiple) can help with the imbalance.
2) I wanted games matches to be similar to other games, not similar to reviews. With one review, a game could show up in multiple buckets.
3) With games with lots ofr eviews, I get more signal with more reviews.
'''
def combine_reviews(df) -> pd.DataFrame:
  return df.groupby("game_name")["review_text"].apply(" ".join).reset_index(name="review")


In [18]:

# This uses the junk patterns to remove words that promote the review, not the game
def remove_boilerplate(text: str) -> str:

    #There are junk patterns causing issues in the clustering. I'm removing those.
    JUNK_PATTERNS = [
        r"english translation",
        r"dutch",
        r"deepl",
        r"please let us know",
        r"opinionated gamers",
        r"original review",
        r"geeklist",
        r"family gamers website",
        r"review.*pictures",
    ]
    lines = text.splitlines()
    keep = []
    for line in lines:
        low = line.lower()
        if any(re.search(p, low) for p in JUNK_PATTERNS):
            continue
        keep.append(line)
    return "\n".join(keep)

In [19]:
# data cleaning/preprocessing is similar but different from what I did earlier.
# I'm still removing stopwords and making the document lower case. However, I'm not filtering for numbers and dates
# I'm also using lemmatization


# This helped with clustering - I'm removing words that don't appear in max_df but are so generic that they don't help you really cluster games based on how they play.
DOMAIN_STOP = {
    "game","games","play","playing","played","player","players",
    "turn","turns","round","rounds","card","cards","deck","hand",
    "rule","rules","component","components","piece","pieces",
    "board","table","points","score","scoring","win","winning",
    "mechanic","mechanics"
}
# Negations might be meaningful so I want to keep those if they were in the stopword_list
negations = {"no", "not", "nor", "never", "n't"}
stopword_list = (set(stopwords.words("english")) - negations) | DOMAIN_STOP

def clean_text(text):
  # Decode HTML entities
  text = html.unescape(text)

  # Remove HTML tags first
  text = re.sub(r"<[^>]+>", "\n", text)   # turn tags into newlines (helps splitlines)
  text = re.sub(r"\bamp\b", " ", text)

  # Now remove boilerplate lines
  text2 = remove_boilerplate(text)
  if text2.strip():        # fallback so we don't wipe everything
      text = text2

  # Remove URLs
  text = re.sub(r"https?://\S+|www\.\S+", " ", text)

  # Normalize spacing/case
  text = re.sub(r"\s+", " ", text.lower()).strip()

  # Tokenize so we can stopword-filter and (optionally) lemmatize
  tokens = tokenizer.tokenize(text)

  # Drop punctuation-only tokens (keep things with letters/digits)
  tokens = [t for t in tokens if re.search(r"[a-z0-9]", t)]

  # lemmatization reduces word-form variants (players/playing -> player/play)
  doc = nlp.make_doc(" ".join(tokens))    # lightweight doc (no parser/NER)
  doc = lemmatizer(doc)
  text = ' '.join([word.lemma_ if word.lemma_ != '-PRON-' else word.text for word in doc])

  # Stopword removal (keeping negations) + remove tiny tokens
  tokens = [t.strip() for t in tokenizer.tokenize(text)]
  tokens = [t for t in tokens if t and t not in stopword_list and len(t) >= 2]
  text = " ".join(tokens)

  return text


In [20]:
# Time to clean the text
def clean_all(df, skip_cleaning = True):
  if skip_cleaning:
    df = pd.read_csv("Cleaned_filed.csv")
  else:
    df = combine_reviews(df)
    texts = df["review"].tolist()
    df["clean_text"] = [clean_text(t) for t in texts]
  return df

In [21]:
project_data.dataframe = clean_all(project_data.dataframe, True)

## Method 0.75: Feature engineering — TF‑IDF → SVD → Normalize (Unit 3)

In [22]:
# Let's vectorize these docs
def tfidf(project_data):
  norm_corpus = project_data.dataframe["clean_text"].tolist()

  # I tried just bi-grams but it did not give good clusters. Unigrams + bi-grams worked better.
  project_data.tv = TfidfVectorizer(
      ngram_range=(1,2),
      min_df=2,
      max_df=0.6,
      sublinear_tf=True
  )
  tv_matrix = project_data.tv.fit_transform(norm_corpus)

  # I wasn't getting great clusters. Chat GPT recommended performing a dimensional reduction to turn the matrix into a smaller and denser representatation
  # This turned a straight line into something I could use
  project_data.svd = TruncatedSVD(n_components=100, random_state=42)  # This reduces the number of dimensions to 100
  X_svd = project_data.svd.fit_transform(tv_matrix)
  project_data.norm = Normalizer(copy=False)  # This scales the document vector and makes the comparison behavior perform somewhat like cosign similarity
  project_data.X = project_data.norm.fit_transform(X_svd)

  return





In [23]:
tfidf(project_data)


## Method 1 — Document clustering (Unit 3)

In [24]:
# Based on the Unit 3 results, k is 12

def cluster(project_data: SavedData, k: int, print_examples = False):

  k = 12

  # Find our clusters based on the ideal number
  km = KMeans(n_clusters=k, random_state=42, n_init=50)
  project_data.dataframe["cluster"] = km.fit_predict(project_data.X)
  project_data.dataframe["cluster_label"] = project_data.dataframe["cluster"].map(project_data.cluster_labels).fillna("Unlabeled")

  if print_examples and project_data.tv is not None:
    # Get the top terms for each cluster
    terms = np.array(project_data.tv.get_feature_names_out())
    centers_svd = km.cluster_centers_              # centers in SVD space
    centers_tfidf = centers_svd @ project_data.svd.components_  # approximate centers in TF-IDF term space

    # Print clusters with representative games
    for c in range(k):
        # get the games in this cluster
        label = project_data.cluster_labels.get(c, 'Unlabeled')
        in_cluster = (project_data.dataframe["cluster"] == c)
        cluster_rows = np.where(in_cluster.values)[0]  # number of games in the cluster

        # Get the top 10 terms for this cluster
        top_term_index = centers_tfidf[c].argsort()[-10:][::-1]
        top_terms = terms[top_term_index]

        mask = project_data.dataframe["cluster"] == c
        idxs = np.where(mask.values)[0]

    # Pick up to 10 "best" games that represent the cluster by computing each games distance from the center
    distances = np.linalg.norm(project_data.X[cluster_rows] - centers_svd[c], axis=1)
    closest_rows = cluster_rows[np.argsort(distances)[:10]]
    example_games = project_data.dataframe.iloc[closest_rows]["game_name"].tolist()

    # Print results
    print(f"\nCluster {c}: {label}. Game count: {in_cluster.sum()}")
    print("Top terms:", ", ".join(top_terms))
    print("Example games:", example_games)

  return

In [25]:
# Fit final KMeans model (Unit 3 used k≈12/13; defaulting to 12)
cluster(project_data, 12, print_examples = True)
project_data.dataframe[["game_name","cluster"]].head()


Cluster 11: Card games. Game count: 72
Top terms: pile, discard, draw, luck, trick, value, face, five, color, quick
Example games: ['Gambit Royale', 'Hey Waiter!', 'Tavarua', 'HMS Dolores', 'Shelfie Stacker', 'Dodekka', 'Via Magica', 'The Battle of Red Cliffs', 'Shanghaien', 'Enchanted Plumes']


,game_name,cluster
0,13 Clues,4
1,1777: The Year of the Hangman,6
2,1812: Napoleon's Fateful March,6
3,1822MX,8
4,18RoyalGorge: The Rails of Fremont County and ...,8


## Method 2 — Document similarity recommendations (Unit 3)

In [26]:
# Second unsupervised method is document similarity so I can recommend games
def create_similarities(project_data):
  doc_sim = cosine_similarity(project_data.X)   # cosine similarity between documents
  doc_sim_df = pd.DataFrame(doc_sim)       # convert to dataframe
  project_data.titles = pd.Series(project_data.dataframe.index.values, index=project_data.dataframe["game_name"]).to_dict()
  project_data.similarity = doc_sim_df
  return


### Compute cosine similarity matrix (on the SVD+normalized vectors)

In [27]:
# Try out the recommendation system
create_similarities(project_data)

## Method 3 — Sentiment analysis (VADER)

In [28]:
def vader_compound(text: str, sia) -> float:
    return sia.polarity_scores(str(text))['compound']

In [29]:
def sentiment_analysis(project_data):
  sia = SentimentIntensityAnalyzer()
  project_data.dataframe['sentiment'] = project_data.dataframe['clean_text'].apply(vader_compound, args=(sia,))
  project_data.dataframe[['game_name','sentiment']].describe()
  return

In [30]:
def sentiment_match(s_game: float, s_target: float) -> float:
    # VADER compound is in [-1, 1], max abs diff is 2
    return 1.0 - (abs(s_game - s_target) / 2.0)

In [31]:

sentiment_analysis(project_data)
display(project_data.dataframe[['game_name', 'sentiment']].head())

,game_name,sentiment
0,13 Clues,0.9964
1,1777: The Year of the Hangman,-0.9969
2,1812: Napoleon's Fateful March,-0.6721
3,1822MX,0.9984
4,18RoyalGorge: The Rails of Fremont County and ...,0.9992


## Putting the recommendations together

In [32]:
# This function is the brains of the recommendation system. It ties everything together
# The function takes our saved project data, a query type (game name or natural language), a query, a weight and cluster to find a new boad game
# Returns a data frame of the top n games that match the search
def get_game_recommendations(project_data: SavedData,
                                 query_type: str,
                                 query_value: str,
                                 sentiment_weight: float = 0.25,
                                 cluster_id: int = None,

    df = project_data.dataframe # Holds the review data
    X = project_data.X # This holds the SVD-normalized document vectors

    sims = None
    target_sent = None
    original_game_idx = -1 # Initialize to an invalid index

    # Search by game name
    if query_type == 'game_name':
        # Find the index of the queried game
        game_indices = df.index[df['game_name'] == query_value]

        # Game not found
        if len(game_indices) == 0:
            raise ValueError(f"Game '{query_value}' not found in the dataset.")

        original_game_idx = int(game_indices[0])

        # Retrieve precomputed similarity scores for this game
        sims = project_data.similarity.iloc[original_game_idx].values

        # Get the sentiment of the queried game
        target_sent = float(df.loc[original_game_idx, 'sentiment'])

        # Exclude the queried game itself from recommendations by setting its similarity to a very low value
        sims[original_game_idx] = -1.0 # Or -np.inf for robust exclusion

    # Natural language search
    elif query_type == 'text_query':
        # Ensure NLP models are available for text embedding
        if project_data.tv is None or project_data.svd is None or project_data.norm is None:
            raise ValueError("TFIDF, SVD, or Normalizer not initialized in project_data. Cannot process text query.")

        # Create an embedder pipeline to transform the text query into the same vector space (SVD-normalized)
        embedder = make_pipeline(project_data.tv, project_data.svd, project_data.norm)
        q_emb = embedder.transform([query_value])

        # Calculate cosine similarity between the query embedding and all game embeddings
        sims = cosine_similarity(q_emb, X).ravel()

        # For text queries, use the median sentiment of the dataset as the target for sentiment matching.
        # This provides a neutral baseline; could be made configurable or derived from query if possible.
        target_sent = float(df['sentiment'].median())

    else:
        raise ValueError("Invalid query_type. Must be 'game_name' or 'text_query'.")

    # Determine the set of candidate games based on cluster filtering
    if cluster_id is not None:
        mask = (df["cluster"].to_numpy() == cluster_id)
        candidate_indices = np.where(mask)[0]
    else:
        # If no cluster_id is specified, all games are candidates
        candidate_indices = np.arange(len(df))

    # Calculate sentiment match scores for all candidate games against the target sentiment
    sents = df['sentiment'].to_numpy(dtype=float)
    sentiment_match_scores = np.array([sentiment_match(s, target_sent) for s in sents])

    # Combine similarity and sentiment match scores for candidate games
    # We use sims[candidate_indices] and sentiment_match_scores[candidate_indices] to get scores for only relevant games
    combined_scores_for_candidates = (1.0 - sentiment_weight) * sims[candidate_indices] + sentiment_weight * sentiment_match_scores[candidate_indices]

    # Get the indices of the top N recommendations from the candidate set, based on combined score
    # np.argsort returns indices that would sort an array, [::-1] reverses to get descending order
    # [:top_n] takes the top N of those indices
    top_candidate_rank_indices = np.argsort(combined_scores_for_candidates)[::-1][:top_n]
    top_recommended_indices = candidate_indices[top_candidate_rank_indices]

    # Create the result DataFrame with relevant information
    result_df = df.iloc[top_recommended_indices][['game_name','cluster','sentiment']].copy()
    result_df['similarity'] = sims[top_recommended_indices]
    result_df['sentiment_match'] = sentiment_match_scores[top_recommended_indices]
    result_df['combined_score'] = combined_scores_for_candidates[top_candidate_rank_indices] # Use the sorted scores for candidates

    # Add human-readable cluster labels
    result_df['cluster_label'] = result_df['cluster'].map(project_data.cluster_labels).fillna("Unlabeled")

    return result_df.sort_values('combined_score', ascending=False).reset_index(drop=True)

In [33]:
recs_gloomhaven = get_game_recommendations(
    project_data,
    query_type='game_name',
    query_value='Gloomhaven',
    sentiment_weight=0.25,
    cluster_id = 2,
    top_n=10
)
display(recs_gloomhaven)


,game_name,cluster,sentiment,similarity,sentiment_match,combined_score,cluster_label
0,Through the Ages: A New Story of Civilization,2,1.0000,0.445289,0.99990,0.583942,Conflict strategy and dudes-on-a-map
1,Innovation,2,1.0000,0.407153,0.99990,0.555340,Conflict strategy and dudes-on-a-map
2,Scythe,2,1.0000,0.369230,0.99990,0.526898,Conflict strategy and dudes-on-a-map
3,Archmage,2,1.0000,0.354429,0.99990,0.515796,Conflict strategy and dudes-on-a-map
4,Blood Bowl (Third Edition),2,0.9999,0.343607,0.99995,0.507693,Conflict strategy and dudes-on-a-map
5,Twilight Struggle,2,0.9975,0.343943,0.99885,0.507670,Conflict strategy and dudes-on-a-map
6,Champions of the Galaxy,2,1.0000,0.342376,0.99990,0.506757,Conflict strategy and dudes-on-a-map
7,Twilight Imperium: Fourth Edition,2,0.9999,0.339226,0.99995,0.504407,Conflict strategy and dudes-on-a-map
8,Ankh: Gods of Egypt,2,1.0000,0.337338,0.99990,0.502979,Conflict strategy and dudes-on-a-map
9,Android: Netrunner,2,1.0000,0.334426,0.99990,0.500794,Conflict strategy and dudes-on-a-map


## Method 4 — LLM explanations

In [34]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('gemini')

In [35]:
def format_recs_for_prompt(seed_game: str, rec_df: pd.DataFrame) -> str:
    lines = [f"Seed game: {seed_game}", "Recommendations:"]
    for r in rec_df.itertuples():
        score = getattr(r, 'score', None)
        score_s = f"{score:.4f}" if isinstance(score, (int,float)) else str(score)
        lines.append(
            f"- {r.game_name} (cluster={getattr(r,'cluster',None)}, sentiment={getattr(r,'sentiment',None):.3f}, score={score_s})"
        )
    return "".join(lines)

In [36]:
def gemini_explain(seed_game: str, rec_df: pd.DataFrame):
  # genai.configure(api_key=GEMINI_API_KEY) # Removed as 'configure' is not in google.genai

  url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?key={GEMINI_API_KEY}"

  headers = {"Content-Type": "application/json"}
  prompt = (
      "You are helping explain board game recommendations. "
      "Give 1-2 concise reasons per recommendation, referencing mechanics/themes implied by text for each listed game."
      " Format te recommendations so the recommended game is shown along with the reason for the recommendation. Do not return an introductory sentence"
      + format_recs_for_prompt(seed_game, rec_df)
  )

  data = {
      "contents": [
          {
              "parts": [{"text": prompt}]
          }
      ]
  }
  response = requests.post(url, headers=headers, data=json.dumps(data))
  if response.status_code == 200:
      return response.json()["candidates"][0]["content"]["parts"][0]["text"]
  else:
      return f"Error: {response.status_code} - {response.text}"


In [37]:
gemini_explanation = gemini_explain('Gloomhaven', recs_gloomhaven)
print(gemini_explanation)

Okay, here are the board game recommendations, along with reasons why they might appeal to someone who enjoys Gloomhaven, focusing on mechanics and themes suggested by the game titles:

*   **Through the Ages: A New Story of Civilization**: Deep strategic card play and engine building, where you develop your civilization through ages of technological advancements and societal shifts.
*   **Innovation**: Card driven civilization game where players race to achieve victory by acquiring innovations and tech.
*   **Scythe**: Engine-building with area control elements and resource management. You are controlling a faction in an alternate 1920s history.
*   **Archmage**: For those who enjoy character progression and the feeling of becoming more powerful over time through magical abilities.
*   **Blood Bowl (Third Edition)**: Tactical combat and team management in a fantasy sports setting, if you enjoy the strategic combat of Gloomhaven but want something more lighthearted.
*   **Twilight Stru

In [38]:
def export_artificats:
  os.makedirs("artifacts", exist_ok=True)

  # Save the main table (include any columns you want to display in the app)
  project_data.dataframe.to_parquet("artifacts/games.parquet", index=False)

  # Save embeddings (your SVD+normalized vectors)
  np.save("artifacts/X.npy", project_data.X)

  # Save fitted models so text queries can be embedded the same way
  joblib.dump(project_data.tv,   "artifacts/tfidf.joblib")
  joblib.dump(project_data.svd,  "artifacts/svd.joblib")
  joblib.dump(project_data.norm, "artifacts/norm.joblib")

  return

SyntaxError: expected '(' (3223994387.py, line 1)

In [ ]:
recommender_code = r'''
from dataclasses import dataclass
import numpy as np
import pandas as pd
import joblib
from sklearn.pipeline import make_pipeline
from sklearn.metrics.pairwise import cosine_similarity

@dataclass
class RecommenderArtifacts:
    df: pd.DataFrame
    X: np.ndarray
    embedder: object
    cluster_labels: dict

def load_artifacts(artifacts_dir: str = "artifacts") -> RecommenderArtifacts:
    df = pd.read_parquet(f"{artifacts_dir}/games.parquet")
    X = np.load(f"{artifacts_dir}/X.npy")

    tv   = joblib.load(f"{artifacts_dir}/tfidf.joblib")
    svd  = joblib.load(f"{artifacts_dir}/svd.joblib")
    norm = joblib.load(f"{artifacts_dir}/norm.joblib")

    embedder = make_pipeline(tv, svd, norm)

    # Keep your cluster label mapping (same as your notebook)
    cluster_labels = {
        0: "Tile-laying and spatial puzzles",
        1: "Air & naval wargames",
        2: "Conflict strategy and dudes-on-a-map",
        3: "Kids & family games",
        4: "Deduction & puzzles",
        5: "Word games",
        6: "Land wargames",
        7: "Dungeon crawler",
        8: "Light filler and abstract games",
        9: "Heavy euro and engine building",
        10: "Midweight tile and action games (mid Euros)",
        11: "Card games",
    }

    return RecommenderArtifacts(df=df, X=X, embedder=embedder, cluster_labels=cluster_labels)

def sentiment_match(s_game: float, s_target: float) -> float:
    # Same logic as your notebook:
    # VADER compound is in [-1, 1], max abs diff is 2
    return 1.0 - (abs(s_game - s_target) / 2.0)

def sims_from_game(df: pd.DataFrame, X: np.ndarray, game_name: str):
    idxs = df.index[df["game_name"] == game_name].tolist()
    if not idxs:
        raise ValueError(f"Game '{game_name}' not found.")
    idx = int(idxs[0])

    sims = cosine_similarity(X[idx:idx+1], X).ravel()
    sims[idx] = -1.0
    target_sent = float(df.loc[idx, "sentiment"])
    return sims, target_sent, idx

def sims_from_text(embedder, X: np.ndarray, text: str):
    q = embedder.transform([text])
    sims = cosine_similarity(q, X).ravel()
    return sims

def get_recommendations(art: RecommenderArtifacts,
                        query_type: str,
                        query_value: str,
                        sentiment_weight: float = 0.25,
                        cluster_id: int | None = None,
                        top_n: int = 10) -> pd.DataFrame:
    df = art.df
    X = art.X

    if query_type == "game_name":
        sims, target_sent, _ = sims_from_game(df, X, query_value)
    elif query_type == "text_query":
        sims = sims_from_text(art.embedder, X, query_value)
        target_sent = float(df["sentiment"].median())
    else:
        raise ValueError("query_type must be 'game_name' or 'text_query'.")

    # Candidate filter
    if cluster_id is not None:
        candidate_indices = np.where(df["cluster"].to_numpy() == cluster_id)[0]
    else:
        candidate_indices = np.arange(len(df))

    sents = df["sentiment"].to_numpy(dtype=float)
    sent_match_scores = np.array([sentiment_match(s, target_sent) for s in sents], dtype=float)

    combined = (1.0 - sentiment_weight) * sims + sentiment_weight * sent_match_scores

    # Rank only candidates
    cand_scores = combined[candidate_indices]
    top_local = np.argsort(cand_scores)[::-1][:top_n]
    top_idx = candidate_indices[top_local]

    out = df.iloc[top_idx].copy()
    out["similarity"] = sims[top_idx]
    out["sentiment_match"] = sent_match_scores[top_idx]
    out["combined_score"] = combined[top_idx]
    out["cluster_label"] = out["cluster"].map(art.cluster_labels).fillna("Unlabeled")

    # nice ordering
    cols = ["game_name", "cluster", "cluster_label", "sentiment", "similarity", "sentiment_match", "combined_score"]
    cols = [c for c in cols if c in out.columns] + [c for c in out.columns if c not in cols]
    out = out[cols].sort_values("combined_score", ascending=False).reset_index(drop=True)
    return out
'''
with open("recommender.py", "w", encoding="utf-8") as f:
    f.write(recommender_code)

print("✅ Wrote recommender.py")